## Neural Machine Translator - Attention

In [ ]:
import re
import random
import math
import unicodedata
from collections import Counter
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from datasets import load_dataset

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", device)

In [ ]:
dataset = load_dataset(
    "Helsinki-NLP/opus_books",
    "en-pt"
)

dataset

In [ ]:
print(dataset)
print(dataset["train"][0])

for i in range(5):
    example = dataset["train"][i]["translation"]

    print("EN:", example["en"])
    print("PT:", example["pt"])
    print("-" * 80)

In [ ]:
# Clean 
def clean_text(text):
    # Unicode normalization
    text = unicodedata.normalize("NFKC", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    # Remove leading/trailing whitespace
    text = text.strip()

    return text
test = "   Hello    world!   "
print(clean_text(test))

In [ ]:
pairs = []

for example in dataset["train"]:
    en = clean_text(example["translation"]["en"])
    pt = clean_text(example["translation"]["pt"])

    if len(en) > 0 and len(pt) > 0:
        pairs.append((en, pt))

print("Number of sentence pairs:", len(pairs))

In [ ]:
# Tokenization
TOKEN_PATTERN = re.compile(
    r"\w+|[^\w\s]",
    flags=re.UNICODE
)


def tokenize(text):
    text = text.lower()
    return TOKEN_PATTERN.findall(text)
sentence = "Hello, how are you?"
print(tokenize(sentence))
print(tokenize("Olá, como você está?"))

for en, pt in pairs[:5]:
    print("EN:", tokenize(en))
    print("PT:", tokenize(pt))
    print()

In [ ]:
class Vocab:
    def __init__(self, token_lists, min_freq=1):
        counter = Counter()

        for tokens in token_lists:
            counter.update(tokens)

        self.special_tokens = [
            "<PAD>",
            "<UNK>",
            "<BOS>",
            "<EOS>"
        ]

        self.itos = list(self.special_tokens)

        for token, freq in counter.items():
            if freq >= min_freq and token not in self.itos:
                self.itos.append(token)

        self.stoi = {
            token: idx
            for idx, token in enumerate(self.itos)
        }

    def __len__(self):
        return len(self.itos)

    def encode(self, tokens):
        unk_id = self.stoi["<UNK>"]

        return [
            self.stoi.get(token, unk_id)
            for token in tokens
        ]

    def decode(self, ids, remove_special=True):
        tokens = []

        for idx in ids:
            token = self.itos[int(idx)]

            if remove_special and token in self.special_tokens:
                continue

            tokens.append(token)

        return tokens

    @property
    def pad_id(self):
        return self.stoi["<PAD>"]

    @property
    def unk_id(self):
        return self.stoi["<UNK>"]

    @property
    def bos_id(self):
        return self.stoi["<BOS>"]

    @property
    def eos_id(self):
        return self.stoi["<EOS>"]

In [ ]:
# Keep the experiment reproducible and avoid extremely long book paragraphs.
# A tiny corpus cannot learn reliable translations for rare words, so map words
# that appear only once in the training split to <UNK>.
MAX_TOKENS = 60
MIN_FREQ = 2

pairs = [
    (en, pt) for en, pt in pairs
    if len(tokenize(en)) <= MAX_TOKENS and len(tokenize(pt)) <= MAX_TOKENS
]

split_rng = random.Random(SEED)
split_rng.shuffle(pairs)
split_idx = int(0.9 * len(pairs))
train_pairs = pairs[:split_idx]
valid_pairs = pairs[split_idx:]

print(f"After length filtering: {len(pairs)} pairs")
print("Train:", len(train_pairs))
print("Validation:", len(valid_pairs))

train_en_tokens = [tokenize(en) for en, _ in train_pairs]
train_pt_tokens = [tokenize(pt) for _, pt in train_pairs]

SRC_VOCAB = Vocab(train_en_tokens, min_freq=MIN_FREQ)
TRG_VOCAB = Vocab(train_pt_tokens, min_freq=MIN_FREQ)

print("English vocabulary:", len(SRC_VOCAB))
print("Portuguese vocabulary:", len(TRG_VOCAB))
print("Special token ids:", SRC_VOCAB.pad_id, SRC_VOCAB.bos_id, SRC_VOCAB.eos_id)


In [ ]:
def numericalize_pair(en, pt):
    src_tokens = tokenize(en)
    trg_tokens = tokenize(pt)

    src_ids = SRC_VOCAB.encode(src_tokens)

    trg_ids = (
        [TRG_VOCAB.bos_id]
        + TRG_VOCAB.encode(trg_tokens)
        + [TRG_VOCAB.eos_id]
    )

    return src_ids, trg_ids

src_ids, trg_ids = numericalize_pair(
    train_pairs[0][0],
    train_pairs[0][1]
)
print(src_ids)
print(trg_ids)

In [ ]:
# Dataset
class TranslationDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        en, pt = self.pairs[idx]

        src_ids, trg_ids = numericalize_pair(en, pt)

        return (
            torch.tensor(src_ids, dtype=torch.long),
            torch.tensor(trg_ids, dtype=torch.long)
        )

train_dataset = TranslationDataset(train_pairs)
valid_dataset = TranslationDataset(valid_pairs)

print(len(train_dataset))
print(len(valid_dataset))

In [ ]:
def collate_fn(batch):
    src_batch, trg_batch = zip(*batch)

    src_batch = pad_sequence(
        src_batch,
        batch_first=True,
        padding_value=SRC_VOCAB.pad_id
    )

    trg_batch = pad_sequence(
        trg_batch,
        batch_first=True,
        padding_value=TRG_VOCAB.pad_id
    )

    return src_batch, trg_batch

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

src, trg = next(iter(train_loader))

print("SRC shape:", src.shape)
print("TRG shape:", trg.shape)

print("SRC:")
print(src[0])

print("TRG:")
print(trg[0])

## NMT Model

In [ ]:
from torch.nn.utils.rnn import (
    pad_sequence,
    pack_padded_sequence,
    pad_packed_sequence
)


class Encoder(nn.Module):
    def __init__(
        self,
        input_dim,
        embedding_dim,
        encoder_hidden_dim,
        padding_idx
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            input_dim,
            embedding_dim,
            padding_idx=padding_idx
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=encoder_hidden_dim,
            batch_first=True,
            bidirectional=True
        )

    def forward(self, src):
        """
        src:
            [batch, src_len]

        returns:
            outputs:
                [batch, src_len, 2 * hidden_dim]

            hidden:
                [2, batch, hidden_dim]

            cell:
                [2, batch, hidden_dim]
        """

        # ------------------------------------------------
        # 1. Create padding mask
        # ------------------------------------------------

        src_mask = src != SRC_VOCAB.pad_id

        # Number of real tokens in each sentence
        src_lengths = src_mask.sum(dim=1)

        # ------------------------------------------------
        # 2. Embedding
        # ------------------------------------------------

        embedded = self.embedding(src)

        # ------------------------------------------------
        # 3. Pack the sequences
        # ------------------------------------------------

        packed_embedded = pack_padded_sequence(
            embedded,
            src_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        # ------------------------------------------------
        # 4. BiLSTM
        # ------------------------------------------------

        packed_outputs, (hidden, cell) = self.lstm(
            packed_embedded
        )

        # ------------------------------------------------
        # 5. Unpack
        # ------------------------------------------------

        outputs, _ = pad_packed_sequence(
            packed_outputs,
            batch_first=True,
            total_length=src.size(1)
        )

        return outputs, hidden, cell

In [ ]:
# Attn model
class ScaledDotProductCrossAttention(nn.Module):
    def __init__(
        self,
        query_dim,
        encoder_dim,
        attention_dim
    ):
        super().__init__()

        self.W_q = nn.Linear(
            query_dim,
            attention_dim,
            bias=False
        )

        self.W_k = nn.Linear(
            encoder_dim,
            attention_dim,
            bias=False
        )

        self.W_v = nn.Linear(
            encoder_dim,
            attention_dim,
            bias=False
        )

        self.scale = math.sqrt(attention_dim)

    def forward(
        self,
        query,
        encoder_outputs,
        src_mask
    ):
        """
        query:
            [batch, query_dim]

        encoder_outputs:
            [batch, src_len, encoder_dim]

        src_mask:
            [batch, src_len]

        returns:
            context:
                [batch, attention_dim]

            attention_weights:
                [batch, src_len]
        """

        Q = self.W_q(query)
        K = self.W_k(encoder_outputs)
        V = self.W_v(encoder_outputs)

        # [batch, src_len]
        scores = torch.bmm(
            K,
            Q.unsqueeze(2)
        ).squeeze(2)

        scores = scores / self.scale

        # Prevent attention from looking at <PAD>
        scores = scores.masked_fill(
            ~src_mask,
            float("-inf")
        )

        attention_weights = F.softmax(
            scores,
            dim=-1
        )

        context = torch.bmm(
            attention_weights.unsqueeze(1),
            V
        ).squeeze(1)

        return context, attention_weights

In [ ]:
class Decoder(nn.Module):
    def __init__(
        self, output_dim, embedding_dim, decoder_hidden_dim,
        encoder_output_dim, attention_dim, padding_idx, dropout=0.25
    ):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, embedding_dim, padding_idx=padding_idx)
        self.dropout = nn.Dropout(dropout)
        self.query_lstm = nn.LSTM(embedding_dim, decoder_hidden_dim, batch_first=True)
        self.attention = ScaledDotProductCrossAttention(
            decoder_hidden_dim, encoder_output_dim, attention_dim
        )
        self.context_projection = nn.Linear(attention_dim, decoder_hidden_dim)
        self.layer_norm = nn.LayerNorm(decoder_hidden_dim)
        self.post_lstm = nn.LSTM(decoder_hidden_dim, decoder_hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(decoder_hidden_dim, output_dim)

    def forward_step(
        self, input_token, query_hidden, query_cell, post_hidden, post_cell,
        encoder_outputs, src_mask
    ):
        embedded = self.dropout(self.embedding(input_token)).unsqueeze(1)
        query_output, (query_hidden, query_cell) = self.query_lstm(
            embedded, (query_hidden, query_cell)
        )
        q_t = query_output.squeeze(1)
        context, attention_weights = self.attention(q_t, encoder_outputs, src_mask)
        fused = self.layer_norm(q_t + self.context_projection(context)).unsqueeze(1)
        post_output, (post_hidden, post_cell) = self.post_lstm(
            fused, (post_hidden, post_cell)
        )
        logits = self.fc_out(self.dropout(post_output.squeeze(1)))
        return logits, query_hidden, query_cell, post_hidden, post_cell, attention_weights


In [ ]:
class Seq2Seq(nn.Module):
    def __init__(
        self,
        encoder,
        decoder,
        encoder_hidden_dim,
        decoder_hidden_dim
    ):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder

        # BiLSTM encoder has two directions.
        # Combine them to initialize the decoder.

        self.hidden_projection = nn.Linear(
            encoder_hidden_dim * 2,
            decoder_hidden_dim
        )

        self.cell_projection = nn.Linear(
            encoder_hidden_dim * 2,
            decoder_hidden_dim
        )

    def initialize_decoder_state(
        self,
        encoder_hidden,
        encoder_cell
    ):
        """
        encoder_hidden:
            [2, batch, encoder_hidden_dim]

        encoder_cell:
            [2, batch, encoder_hidden_dim]
        """

        # Concatenate forward and backward states
        hidden = torch.cat(
            [encoder_hidden[0], encoder_hidden[1]],
            dim=1
        )

        cell = torch.cat(
            [encoder_cell[0], encoder_cell[1]],
            dim=1
        )

        hidden = torch.tanh(
            self.hidden_projection(hidden)
        )

        cell = torch.tanh(
            self.cell_projection(cell)
        )

        # LSTM expects:
        # [num_layers, batch, hidden]
        hidden = hidden.unsqueeze(0)
        cell = cell.unsqueeze(0)

        return hidden, cell

    def forward(
        self,
        src,
        trg,
        teacher_forcing_ratio=0.5
    ):
        """
        src:
            [batch, src_len]

        trg:
            [batch, trg_len]
        """

        batch_size = src.size(0)
        trg_len = trg.size(1)

        output_dim = self.decoder.fc_out.out_features

        outputs = torch.zeros(
            batch_size,
            trg_len,
            output_dim,
            device=src.device
        )

        # ------------------------------------------------
        # Encoder
        # ------------------------------------------------

        encoder_outputs, encoder_hidden, encoder_cell = self.encoder(src)

        # ------------------------------------------------
        # Decoder initial states
        # ------------------------------------------------

        query_hidden, query_cell = self.initialize_decoder_state(
            encoder_hidden,
            encoder_cell
        )

        # Start post-attention LSTM from zeros
        post_hidden = torch.zeros_like(query_hidden)
        post_cell = torch.zeros_like(query_cell)

        # ------------------------------------------------
        # Source padding mask
        # ------------------------------------------------

        src_mask = src != SRC_VOCAB.pad_id

        # First decoder input is <BOS>
        input_token = trg[:, 0]

        # ------------------------------------------------
        # Autoregressive decoding
        # ------------------------------------------------

        for t in range(1, trg_len):

            (
                output,
                query_hidden,
                query_cell,
                post_hidden,
                post_cell,
                _
            ) = self.decoder.forward_step(
                input_token,
                query_hidden,
                query_cell,
                post_hidden,
                post_cell,
                encoder_outputs,
                src_mask
            )

            outputs[:, t] = output

            teacher_force = (
                random.random()
                < teacher_forcing_ratio
            )

            best_guess = output.argmax(1)

            input_token = (
                trg[:, t]
                if teacher_force
                else best_guess
            )

        return outputs

In [ ]:
# A smaller model plus dropout is a better fit for roughly one thousand pairs.
INPUT_DIM = len(SRC_VOCAB)
OUTPUT_DIM = len(TRG_VOCAB)
EMBEDDING_DIM = 128
ENCODER_HIDDEN_DIM = 192
DECODER_HIDDEN_DIM = 192
ATTENTION_DIM = 192
DROPOUT = 0.25

print("INPUT_DIM:", INPUT_DIM)
print("OUTPUT_DIM:", OUTPUT_DIM)

encoder = Encoder(INPUT_DIM, EMBEDDING_DIM, ENCODER_HIDDEN_DIM, SRC_VOCAB.pad_id)
decoder = Decoder(
    OUTPUT_DIM, EMBEDDING_DIM, DECODER_HIDDEN_DIM,
    ENCODER_HIDDEN_DIM * 2, ATTENTION_DIM, TRG_VOCAB.pad_id, dropout=DROPOUT
)
model = Seq2Seq(encoder, decoder, ENCODER_HIDDEN_DIM, DECODER_HIDDEN_DIM).to(device)
print(model)


In [ ]:
def count_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


print(
    f"Trainable parameters: "
    f"{count_parameters(model):,}"
)

PAD_IDX = TRG_VOCAB.pad_id
criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_IDX
)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2
)

def _loss_and_tokens(output, trg, criterion):
    output = output[:, 1:].reshape(-1, output.size(-1))
    targets = trg[:, 1:].reshape(-1)
    return criterion(output, targets), (targets != PAD_IDX).sum().item()

def train_epoch(model, loader, optimizer, criterion, teacher_forcing_ratio, clip=1.0):
    model.train()
    total_loss, total_tokens = 0.0, 0
    for src, trg in loader:
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()
        output = model(src, trg, teacher_forcing_ratio=teacher_forcing_ratio)
        loss, token_count = _loss_and_tokens(output, trg, criterion)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        total_loss += loss.item() * token_count
        total_tokens += token_count
    return total_loss / total_tokens

def evaluate(model, loader, criterion):
    """Teacher-forced NLL: stable, comparable validation loss."""
    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for src, trg in loader:
            src, trg = src.to(device), trg.to(device)
            output = model(src, trg, teacher_forcing_ratio=1.0)
            loss, token_count = _loss_and_tokens(output, trg, criterion)
            total_loss += loss.item() * token_count
            total_tokens += token_count
    return total_loss / total_tokens


In [ ]:
N_EPOCHS = 40
PATIENCE = 6
train_losses, valid_losses = [], []
best_valid_loss = float("inf")
epochs_without_improvement = 0

for epoch in range(N_EPOCHS):
    # Start with reliable teacher forcing, then gradually expose the decoder
    # to its own predictions.
    teacher_forcing_ratio = max(0.5, 1.0 - 0.02 * epoch)
    train_loss = train_epoch(
        model, train_loader, optimizer, criterion, teacher_forcing_ratio
    )
    valid_loss = evaluate(model, valid_loader, criterion)
    scheduler.step(valid_loss)
    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

    if valid_loss < best_valid_loss - 1e-4:
        best_valid_loss = valid_loss
        epochs_without_improvement = 0
        torch.save(model.state_dict(), "nmt_attention_best.pt")
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch + 1:02d}/{N_EPOCHS} | TF: {teacher_forcing_ratio:.2f} | "
        f"Train NLL: {train_loss:.4f} | Valid NLL: {valid_loss:.4f} | "
        f"LR: {optimizer.param_groups[0]['lr']:.1e}"
    )
    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping: no validation improvement for {PATIENCE} epochs.")
        break

print(f"Best validation NLL: {best_valid_loss:.4f}")


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    train_losses,
    label="Train"
)

plt.plot(
    valid_losses,
    label="Validation"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("NMT Training and Validation Loss")

plt.legend()
plt.grid()

plt.show()

In [ ]:
def encode_source_sentence(sentence):
    tokens = tokenize(sentence)

    ids = SRC_VOCAB.encode(tokens)

    tensor = torch.tensor(
        ids,
        dtype=torch.long,
        device=device
    ).unsqueeze(0)

    return tensor

In [ ]:
def greedy_translate(
    model,
    sentence,
    max_len=50
):
    model.eval()

    src = encode_source_sentence(sentence)

    with torch.no_grad():

        encoder_outputs, encoder_hidden, encoder_cell = (
            model.encoder(src)
        )

        query_hidden, query_cell = (
            model.initialize_decoder_state(
                encoder_hidden,
                encoder_cell
            )
        )

        post_hidden = torch.zeros_like(query_hidden)
        post_cell = torch.zeros_like(query_cell)

        src_mask = src != SRC_VOCAB.pad_id

        input_token = torch.tensor(
            [TRG_VOCAB.bos_id],
            device=device
        )

        generated_tokens = []

        for _ in range(max_len):

            (
                output,
                query_hidden,
                query_cell,
                post_hidden,
                post_cell,
                _
            ) = model.decoder.forward_step(
                input_token,
                query_hidden,
                query_cell,
                post_hidden,
                post_cell,
                encoder_outputs,
                src_mask
            )

            next_token = output.argmax(
                dim=1
            ).item()

            if next_token == TRG_VOCAB.eos_id:
                break

            generated_tokens.append(next_token)

            input_token = torch.tensor(
                [next_token],
                device=device
            )

    tokens = TRG_VOCAB.decode(
        generated_tokens
    )

    return " ".join(tokens)

In [ ]:
model.load_state_dict(
    torch.load(
        "nmt_attention_best.pt",
        map_location=device
    )
)

sentence = "I love you."

translation = greedy_translate(
    model,
    sentence
)

print("English:", sentence)
print("Portuguese:", translation)

## Held-out translation quality


In [ ]:
def corpus_bleu(references, hypotheses, max_order=4):
    """Small dependency-free corpus BLEU implementation for one reference each."""
    clipped, totals = [0] * max_order, [0] * max_order
    reference_length = hypothesis_length = 0
    for reference, hypothesis in zip(references, hypotheses):
        reference_length += len(reference)
        hypothesis_length += len(hypothesis)
        for order in range(1, max_order + 1):
            reference_ngrams = Counter(
                tuple(reference[i:i + order]) for i in range(max(0, len(reference) - order + 1))
            )
            hypothesis_ngrams = Counter(
                tuple(hypothesis[i:i + order]) for i in range(max(0, len(hypothesis) - order + 1))
            )
            clipped[order - 1] += sum((reference_ngrams & hypothesis_ngrams).values())
            totals[order - 1] += max(0, len(hypothesis) - order + 1)
    if hypothesis_length == 0:
        return 0.0
    # Add-one smoothing keeps BLEU meaningful on this very small validation set.
    precisions = [(match + 1) / (total + 1) for match, total in zip(clipped, totals)]
    brevity_penalty = min(1.0, math.exp(1 - reference_length / hypothesis_length))
    return brevity_penalty * math.exp(sum(math.log(p) for p in precisions) / max_order) * 100

def evaluate_generation(model, pairs, limit=None):
    subset = pairs if limit is None else pairs[:limit]
    references = [tokenize(pt) for _, pt in subset]
    hypotheses = [tokenize(greedy_translate(model, en)) for en, _ in subset]
    return corpus_bleu(references, hypotheses), hypotheses

validation_bleu, validation_hypotheses = evaluate_generation(model, valid_pairs)
print(f"Held-out greedy BLEU-4: {validation_bleu:.2f}")
for (source, reference), hypothesis in zip(valid_pairs[:5], validation_hypotheses[:5]):
    print(f"\nEN:  {source}")
    print(f"REF: {reference}")
    print(f"HYP: {' '.join(hypothesis)}")


In [ ]:
def sample_from_logits(
    logits,
    temperature=1.0
):
    logits = logits / temperature

    probabilities = F.softmax(
        logits,
        dim=-1
    )

    token = torch.multinomial(
        probabilities,
        num_samples=1
    )

    return token.item()

In [ ]:
def sample_translate(
    model,
    sentence,
    temperature=1.0,
    max_len=50
):
    model.eval()

    src = encode_source_sentence(sentence)

    with torch.no_grad():

        encoder_outputs, encoder_hidden, encoder_cell = (
            model.encoder(src)
        )

        query_hidden, query_cell = (
            model.initialize_decoder_state(
                encoder_hidden,
                encoder_cell
            )
        )

        post_hidden = torch.zeros_like(query_hidden)
        post_cell = torch.zeros_like(query_cell)

        src_mask = src != SRC_VOCAB.pad_id

        input_token = torch.tensor(
            [TRG_VOCAB.bos_id],
            device=device
        )

        generated_tokens = []

        for _ in range(max_len):

            (
                output,
                query_hidden,
                query_cell,
                post_hidden,
                post_cell,
                _
            ) = model.decoder.forward_step(
                input_token,
                query_hidden,
                query_cell,
                post_hidden,
                post_cell,
                encoder_outputs,
                src_mask
            )

            next_token = sample_from_logits(
                output,
                temperature=temperature
            )

            if next_token == TRG_VOCAB.eos_id:
                break

            generated_tokens.append(next_token)

            input_token = torch.tensor(
                [next_token],
                device=device
            )

    return " ".join(
        TRG_VOCAB.decode(generated_tokens)
    )

In [ ]:
sentence = "I love you."

for temperature in [0.3, 0.7, 1.0, 1.5]:
    print(
        f"T={temperature}: "
        f"{sample_translate(model, sentence, temperature)}"
    )

In [ ]:
def translate(
    model,
    sentence,
    method="greedy",
    temperature=1.0,
    max_len=50
):
    if method == "greedy":
        return greedy_translate(
            model,
            sentence,
            max_len=max_len
        )

    elif method == "sampling":
        return sample_translate(
            model,
            sentence,
            temperature=temperature,
            max_len=max_len
        )

    else:
        raise ValueError(
            "method must be 'greedy' or 'sampling'"
        )


print(
    translate(
        model,
        "I love you.",
        method="greedy"
    )
)

print(
    translate(
        model,
        "I love you.",
        method="sampling",
        temperature=0.8
    )
)

In [ ]:
def translate_with_attention(
    model,
    sentence,
    max_len=50
):
    model.eval()

    src_tokens = tokenize(sentence)

    src = encode_source_sentence(sentence)

    attention_history = []

    with torch.no_grad():

        encoder_outputs, encoder_hidden, encoder_cell = (
            model.encoder(src)
        )

        query_hidden, query_cell = (
            model.initialize_decoder_state(
                encoder_hidden,
                encoder_cell
            )
        )

        post_hidden = torch.zeros_like(query_hidden)
        post_cell = torch.zeros_like(query_cell)

        src_mask = src != SRC_VOCAB.pad_id

        input_token = torch.tensor(
            [TRG_VOCAB.bos_id],
            device=device
        )

        generated_tokens = []

        for _ in range(max_len):

            (
                output,
                query_hidden,
                query_cell,
                post_hidden,
                post_cell,
                attention_weights
            ) = model.decoder.forward_step(
                input_token,
                query_hidden,
                query_cell,
                post_hidden,
                post_cell,
                encoder_outputs,
                src_mask
            )

            next_token = output.argmax(
                dim=1
            ).item()

            attention_history.append(
                attention_weights.squeeze(0).cpu().numpy()
            )

            if next_token == TRG_VOCAB.eos_id:
                break

            generated_tokens.append(next_token)

            input_token = torch.tensor(
                [next_token],
                device=device
            )

    translated_tokens = TRG_VOCAB.decode(
        generated_tokens
    )

    attention_matrix = np.array(
        attention_history
    )

    return (
        translated_tokens,
        src_tokens,
        attention_matrix
    )

In [ ]:
translated_tokens, src_tokens, attention_matrix = (
    translate_with_attention(
        model,
        "I love you."
    )
)

print(
    "Translation:",
    " ".join(translated_tokens)
)

plt.figure(figsize=(10, 6))

plt.imshow(
    attention_matrix,
    aspect="auto"
)

plt.xticks(
    range(len(src_tokens)),
    src_tokens,
    rotation=45
)

plt.yticks(
    range(len(translated_tokens)),
    translated_tokens
)

plt.xlabel("English source tokens")
plt.ylabel("Generated Portuguese tokens")
plt.title("Cross-Attention")

plt.colorbar(label="Attention weight")

plt.tight_layout()
plt.show()

In [ ]:
def generate_samples(
    model,
    sentence,
    num_samples=20,
    temperature=1.0,
    max_len=50
):
    samples = []

    for _ in range(num_samples):

        translation = sample_translate(
            model,
            sentence,
            temperature=temperature,
            max_len=max_len
        )

        samples.append(translation)

    return samples

In [ ]:
sentence = "I love you."

samples = generate_samples(
    model,
    sentence,
    num_samples=20,
    temperature=0.8
)

for i, sample in enumerate(samples):
    print(f"{i:02d}: {sample}")

In [ ]:
def jaccard_similarity(
    sentence_a,
    sentence_b
):
    a = set(tokenize(sentence_a))
    b = set(tokenize(sentence_b))

    if not a and not b:
        return 1.0

    if not a or not b:
        return 0.0

    return len(a & b) / len(a | b)

print(
    jaccard_similarity(
        "eu amo você",
        "eu gosto você"
    )
)

In [ ]:
def rouge1_f1(
    sentence_a,
    sentence_b
):
    tokens_a = tokenize(sentence_a)
    tokens_b = tokenize(sentence_b)

    if not tokens_a or not tokens_b:
        return 0.0

    counter_a = Counter(tokens_a)
    counter_b = Counter(tokens_b)

    overlap = sum(
        (counter_a & counter_b).values()
    )

    precision = overlap / len(tokens_a)
    recall = overlap / len(tokens_b)

    if precision + recall == 0:
        return 0.0

    return (
        2 * precision * recall
        / (precision + recall)
    )


print(
    rouge1_f1(
        "eu amo você",
        "eu gosto você"
    )
)



In [ ]:
def average_overlap(
    sentence_a,
    sentence_b
):
    jaccard = jaccard_similarity(
        sentence_a,
        sentence_b
    )

    rouge = rouge1_f1(
        sentence_a,
        sentence_b
    )

    return (jaccard + rouge) / 2

In [ ]:
def build_utility_matrix(samples):
    n = len(samples)

    matrix = np.zeros(
        (n, n),
        dtype=np.float32
    )

    for i in range(n):
        for j in range(n):

            matrix[i, j] = average_overlap(
                samples[i],
                samples[j]
            )

    return matrix

In [ ]:
utility_matrix = build_utility_matrix(
    samples
)

plt.figure(figsize=(8, 6))

plt.imshow(
    utility_matrix,
    aspect="auto"
)

plt.xlabel("Candidate j")
plt.ylabel("Candidate i")
plt.title("Candidate Utility Matrix")

plt.colorbar(
    label="Average overlap"
)

plt.show()

In [ ]:
def mbr_decode(
    samples,
    utility_matrix=None
):
    if len(samples) == 0:
        return ""

    if utility_matrix is None:
        utility_matrix = build_utility_matrix(
            samples
        )

    scores = utility_matrix.mean(
        axis=1
    )

    best_idx = np.argmax(scores)

    return (
        samples[best_idx],
        scores,
        best_idx
    )

In [ ]:
best_translation, scores, best_idx = (
    mbr_decode(
        samples,
        utility_matrix
    )
)

print("Candidates:")
for i, sample in enumerate(samples):
    print(
        f"{i:02d} | "
        f"score={scores[i]:.4f} | "
        f"{sample}"
    )

print("\nMBR selected:")
print(best_translation)

In [ ]:
sentence = "I love you."

greedy = translate(
    model,
    sentence,
    method="greedy"
)

samples = generate_samples(
    model,
    sentence,
    num_samples=30,
    temperature=0.8
)

mbr_translation, scores, best_idx = (
    mbr_decode(samples)
)

print("SOURCE")
print(sentence)

print("\nGREEDY")
print(greedy)

print("\nMBR")
print(mbr_translation)

print("\nSAMPLES")
for i, sample in enumerate(samples):
    print(f"{i:02d}: {sample}")

**Note:** This is just RNN fused with Scaled Dot Product Cross Attention, not a utilization of mordern [Transformer](https://arxiv.org/abs/1706.03762), so the result of the model will not be as good as expected

## Production-quality translation: pretrained Transformer

The attention model above is a from-scratch learning exercise. Its training set has only about 1,400 book-sentence pairs, which cannot support useful open-domain translation. This section uses a pretrained Transformer with SentencePiece subwords and evaluates it on the same held-out split.


In [ ]:
# Install the small runtime dependency automatically if this Kaggle image lacks it.
try:
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "transformers", "sentencepiece"
    ])
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

PRETRAINED_MODEL = "Helsinki-NLP/opus-mt-en-ROMANCE"
TARGET_LANGUAGE = "pt"

translation_tokenizer = AutoTokenizer.from_pretrained(PRETRAINED_MODEL)
translation_model = AutoModelForSeq2SeqLM.from_pretrained(PRETRAINED_MODEL).to(device)
translation_model.eval()

print(f"Loaded {PRETRAINED_MODEL} on {device}")


In [ ]:
def pretrained_translate_batch(sentences, batch_size=16, max_new_tokens=128):
    # Translate English text to Portuguese in batches with deterministic beam search.
    translations = []
    translation_model.eval()
    for start in range(0, len(sentences), batch_size):
        batch = sentences[start:start + batch_size]
        # This multilingual Marian model requires a target-language prefix.
        prompts = [f">>{TARGET_LANGUAGE}<< {sentence}" for sentence in batch]
        encoded = translation_tokenizer(
            prompts, return_tensors="pt", padding=True, truncation=True, max_length=128
        ).to(device)
        with torch.inference_mode():
            generated = translation_model.generate(
                **encoded, num_beams=4, max_new_tokens=max_new_tokens,
                early_stopping=True
            )
        translations.extend(translation_tokenizer.batch_decode(generated, skip_special_tokens=True))
    return translations

def pretrained_translate(sentence):
    return pretrained_translate_batch([sentence])[0]

print(pretrained_translate("I love you."))


In [ ]:
# Fair comparison: use exactly the held-out pairs used by the attention baseline.
validation_sources = [en for en, _ in valid_pairs]
validation_references = [tokenize(pt) for _, pt in valid_pairs]
pretrained_hypotheses = [tokenize(text) for text in pretrained_translate_batch(validation_sources)]
pretrained_bleu = corpus_bleu(validation_references, pretrained_hypotheses)

print(f"Pretrained held-out greedy BLEU-4: {pretrained_bleu:.2f}")
print(f"Scratch attention held-out greedy BLEU-4: {validation_bleu:.2f}")
for (source, reference), hypothesis in zip(valid_pairs[:5], pretrained_hypotheses[:5]):
    print(f"\nEN:  {source}")
    print(f"REF: {reference}")
    print(f"HYP: {' '.join(hypothesis)}")


## Acknowledgement:
Thanks [Helsinki-NLP](https://huggingface.co/Helsinki-NLP) for both the data and the pretrained model

The Trained version is available on [kaggle](https://www.kaggle.com/code/cartethiya/notebookef182ce5c8)